# MAFDET gatherer: training the small shared model

The gatherer is a small model (Qwen3-0.6B by default) that, while the user types, writes **material for the
answer**: one line per key term, name or number already typed, with what it is and the facts it knows about it.
On Enter any large model gets that material as text, so it has less reasoning left to do.

This notebook trains it in three parts. Run the cells in order; **restart the runtime between parts**
(Runtime → Restart session): each part installs different libraries and frees the GPU for the next one.
Everything is saved to Google Drive as it goes, so a disconnected session loses at most one batch.

1. **Data.** A large teacher model sees each message *whole* and, for every prefix the user would type on the
   way, writes the lines the gatherer should add at that point, using only the words already typed
   (hindsight: it knows where the message is going, so it picks what will be useful).
2. **Training.** Fine-tuning of the gatherer on those examples (every weight on a GPU with more than 70 GB,
   a LoRA adapter otherwise), loss on the gatherer's lines only.
3. **Export.** Merge, convert to GGUF (Q8_0) for llama.cpp, save to Drive.

Times on an A100 with the defaults (8,000 messages, Qwen3-32B as teacher, reasoning on): about 1.5-3 hours
for the data, 30-60 min for training, 5 min for export. With `TEACHER_THINKING = False` the data take
about a third of the time; on an L4 or T4 everything takes about three times as long. Every part resumes
where it stopped, so the data can be generated over several sessions.

## 0. Setup (run at the start of every part)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
WORK = '/content/drive/MyDrive/mafdet-gatherer'      # everything is saved here
os.makedirs(WORK, exist_ok=True)
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'],
                     capture_output=True, text=True).stdout)

In [ ]:
# The gatherer and its format. MAFDET builds exactly these prompts at run time: keep them in sync.
# The gatherer runs on the user's PC next to the large model, and every MB of VRAM it takes moves part of
# the large model to the CPU. So: Qwen3-0.6B for 6-8 GB cards (RTX 2060, about 0.9 GB with its cache),
# Qwen3-1.7B for 16 GB cards (RTX 5070 Ti), Qwen3-4B only if it fits next to the large model. The data
# from part 1 serve them all: to train another one, change STUDENT and run parts 2 and 3 again.
STUDENT = 'Qwen/Qwen3-0.6B'
N_MESSAGES = 8000                     # messages to generate examples from (each gives 2 to 5 examples)
TEACHER_THINKING = True               # the teacher reasons before writing the material: better data, ~3x the time
ITALIAN_SHARE = 0.4                   # share of messages translated into Italian first

GATHER_SEED = ('<think>\nMaterial for the answer, gathered while the user types: for each key term, name or '
               'number already written, one line with what it is and the facts I know about it, or a partial '
               'result.\n\n')
STEP_HEADER = 'New words: "{new}"\n'  # before the lines of each step; the first step names the whole prefix
END = '<|im_end|>'                    # the gatherer ends a step with it: nothing more to add for now

## Part 1: data

In [ ]:
!pip install -q vllm datasets

In [ ]:
# User messages: open-ended requests (OpenAssistant), word problems (GSM8K), code requests (MBPP).
import json, random, re
from datasets import load_dataset

rng = random.Random(0)
oa = [r['text'].strip() for r in load_dataset('OpenAssistant/oasst2', split='train')
      if r['role'] == 'prompter' and r['parent_id'] is None and r['lang'] in ('en', 'it')
      and 40 <= len(r['text']) <= 500]
gsm = [r['question'].strip() for r in load_dataset('openai/gsm8k', 'main', split='train')]
mbpp = [r['text'].strip() for r in load_dataset('google-research-datasets/mbpp', 'full', split='train')]
rng.shuffle(oa); rng.shuffle(gsm); rng.shuffle(mbpp)
n_oa, n_gsm = int(N_MESSAGES * 0.6), int(N_MESSAGES * 0.2)
messages = ([{'text': t, 'src': 'oasst2'} for t in oa[:n_oa]]
            + [{'text': t, 'src': 'gsm8k'} for t in gsm[:n_gsm]]
            + [{'text': t, 'src': 'mbpp'} for t in mbpp[:N_MESSAGES - n_oa - n_gsm]])
rng.shuffle(messages)
print(len(messages), 'messages;', {s: sum(m['src'] == s for m in messages) for s in ('oasst2', 'gsm8k', 'mbpp')})

In [ ]:
# The teacher: the largest Qwen3 that fits this GPU, served by vLLM.
import torch
from vllm import LLM, SamplingParams

props = torch.cuda.get_device_properties(0)
vram = props.total_memory / 2**30
TEACHER = 'Qwen/Qwen3-32B' if vram > 70 else 'Qwen/Qwen3-32B-AWQ' if vram > 30 else 'Qwen/Qwen3-14B-AWQ'
dtype = 'auto' if props.major >= 8 else 'half'      # the T4 has no bfloat16
print('teacher:', TEACHER, f'({vram:.0f} GB, dtype {dtype})')
llm = LLM(TEACHER, max_model_len=8192, gpu_memory_utilization=0.9, dtype=dtype)
ttok = llm.get_tokenizer()

def ask(prompts, max_tokens=1500, temperature=0.3, thinking=False):
    """The teacher's replies, without its reasoning."""
    texts = [ttok.apply_chat_template([{'role': 'user', 'content': p}], tokenize=False,
                                      add_generation_prompt=True, enable_thinking=thinking) for p in prompts]
    params = SamplingParams(temperature=0.6 if thinking else temperature, top_p=0.95 if thinking else 0.9,
                            max_tokens=max_tokens)
    return [o.outputs[0].text.split('</think>')[-1].strip() for o in llm.generate(texts, params)]

In [ ]:
# A share of the messages in Italian, as a person would type them in a chat.
TRANSLATE = ('Translate this message into natural Italian, as a person would type it in a chat. Keep code, '
             'numbers and names unchanged. Reply with the translation only.\n\n<message>\n{m}\n</message>')
idx = [i for i in range(len(messages)) if rng.random() < ITALIAN_SHARE]
for i, t in zip(idx, ask([TRANSLATE.format(m=messages[i]['text']) for i in idx], max_tokens=800)):
    if t and len(t) < 2 * len(messages[i]['text']) + 50:
        messages[i]['text'], messages[i]['lang'] = t, 'it'
print(sum(m.get('lang') == 'it' for m in messages), 'messages in Italian')

In [ ]:
# The prefixes the user would type on the way: whole words, 2 to 5 of them, the last one the full message.
def prefixes(text, rng):
    words = text.split(' ')
    if len(words) < 6:
        return [text]
    k = min(5, max(2, len(words) // 4))
    cuts = sorted(rng.sample(range(2, len(words)), k - 1))
    return [' '.join(words[:c]) for c in cuts] + [text]

TEACHER_PROMPT = """You are preparing training data for a small model that gathers material for an answer while a user is still typing their message.

The complete message (the small model never sees it whole, only growing):
<message>
{message}
</message>

The small model sees these successive versions, each one a prefix of the message:
{versions}

For each version, write the lines of material the small model should add at that point. Rules:
- Use ONLY words already present in that version. Never use anything from the part not typed yet. You know where the message is going: use that only to choose WHICH of the terms, names and numbers already typed are worth explaining.
- Each line: "- <term, name or number>: <what it is, a fact useful for the final answer, or a partial result>". One short line, at most 25 words.
- Never write about the user, their intentions, or the message being unfinished.
- Do not repeat material from earlier versions. If a version adds nothing worth noting, give an empty list.
- At most 5 lines per version, in English.

Reply with JSON only: {{"steps": [["- line", "- line"], [], ...]}}, exactly {n} lists, one per version, in order."""

STOP = set('about after again also another because before being between could does doing during every from have having here into just more most other over same should some such than that their them then there these they this those through under very what when where which while with would your'.split())

def content_words(text):
    return {w for w in re.findall(r'\w+', text.lower()) if len(w) >= 5 and w not in STOP}

def leaks(line, prefix, message):
    """A word of the message that is not in the prefix yet: the line uses the future."""
    future = content_words(message) - content_words(prefix)
    return bool(content_words(line) & future)

def parse(raw, n):
    try:
        steps = json.loads(raw[raw.index('{'):raw.rindex('}') + 1])['steps']
    except (ValueError, KeyError, TypeError):
        return None
    if not isinstance(steps, list) or len(steps) != n:
        return None
    return [[l.strip() for l in s if isinstance(l, str) and l.strip().startswith('- ')] for s in steps]

In [ ]:
# Generate in batches, saving each one to Drive: a dropped session resumes where it stopped.
OUT = f'{WORK}/data.jsonl'
done = sum(1 for _ in open(OUT)) if os.path.exists(OUT) else 0
print('already done:', done)
stats = {'ok': 0, 'bad_json': 0, 'lines': 0, 'leaks_dropped': 0}
BATCH = 250
for start in range(done, len(messages), BATCH):
    batch = messages[start:start + BATCH]
    pre = [prefixes(m['text'], random.Random(start + j)) for j, m in enumerate(batch)]
    prompts = [TEACHER_PROMPT.format(message=m['text'], n=len(p),
                                     versions='\n'.join(f'Version {i + 1}: "{v}"' for i, v in enumerate(p)))
               for m, p in zip(batch, pre)]
    with open(OUT, 'a') as f:
        replies = ask(prompts, max_tokens=6000 if TEACHER_THINKING else 1500, thinking=TEACHER_THINKING)
        for m, p, raw in zip(batch, pre, replies):
            steps = parse(raw, len(p))
            if steps is None:
                stats['bad_json'] += 1
                f.write(json.dumps({'skip': True}) + '\n')     # keeps the count aligned for resuming
                continue
            kept = []
            for v, lines in zip(p, steps):
                good = [l for l in lines if not leaks(l, v, m['text'])]
                stats['leaks_dropped'] += len(lines) - len(good)
                stats['lines'] += len(good)
                kept.append(good)
            stats['ok'] += 1
            f.write(json.dumps({**m, 'prefixes': p, 'steps': kept}, ensure_ascii=False) + '\n')
    print(start + len(batch), '/', len(messages), stats)

In [ ]:
# Look at a few examples before training: this is where quality is decided.
rows = [json.loads(l) for l in open(OUT)]
rows = [r for r in rows if not r.get('skip')]
print(len(rows), 'messages with examples')
for r in random.Random(1).sample(rows, 3):
    print('=' * 80)
    for v, lines in zip(r['prefixes'], r['steps']):
        print(f'>> "{v}"')
        print('\n'.join(lines) or '   (nothing)')

## Part 2: training (restart the runtime first, then run the two setup cells)

In [ ]:
# torchao is not needed here, and an old copy (preinstalled by Colab or pulled in by vLLM) makes PEFT
# refuse to load ("Found an incompatible version of torchao").
!pip uninstall -y -q torchao
!pip install -q -U transformers peft accelerate

In [ ]:
import json, random, torch
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(STUDENT)
MAX_LEN = 2048

def examples(rec):
    """One example per step: the prompt MAFDET would build at that point, and the lines to add."""
    out, notes, prev = [], '', ''
    for p, lines in zip(rec['prefixes'], rec['steps']):
        new = p[len(os.path.commonprefix([prev, p])):].strip()
        head = STEP_HEADER.format(new=new[-300:])
        prompt = (tok.apply_chat_template([{'role': 'user', 'content': p}], tokenize=False,
                                          add_generation_prompt=True, enable_thinking=True)
                  + GATHER_SEED + notes + head)
        body = ''.join(l + '\n' for l in lines)
        out.append((prompt, body + END))
        notes += head + body + '\n'
        prev = p
    return out

def encode(prompt, completion):
    p = tok(prompt, add_special_tokens=False).input_ids
    c = tok(completion, add_special_tokens=False).input_ids
    if len(p) + len(c) > MAX_LEN:
        return None
    return {'input_ids': p + c, 'labels': [-100] * len(p) + c}     # loss on the gatherer's lines only

rows = [json.loads(l) for l in open(f'{WORK}/data.jsonl')]
rows = [r for r in rows if not r.get('skip')]
random.Random(0).shuffle(rows)
n_eval = max(20, len(rows) // 30)
held_out, train_rows = rows[:n_eval], rows[n_eval:]
train = [e for r in train_rows for e in (encode(*x) for x in examples(r)) if e]
evals = [e for r in held_out for e in (encode(*x) for x in examples(r)) if e]
print(len(train), 'training examples,', len(evals), 'held out')
print(examples(train_rows[0])[1][0][-600:] + '|||' + examples(train_rows[0])[1][1])

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments

bf16 = torch.cuda.is_bf16_supported()
vram = torch.cuda.get_device_properties(0).total_memory / 2**30
FULL = bf16 and vram > 70             # big GPU: train every weight; otherwise a LoRA adapter
NAME = STUDENT.split('/')[-1]
print('student:', NAME, '| full fine-tuning' if FULL else '| LoRA', f'| {vram:.0f} GB')
model = AutoModelForCausalLM.from_pretrained(STUDENT, torch_dtype=torch.bfloat16 if bf16 else torch.float32)
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
if not FULL:
    model = get_peft_model(model, LoraConfig(
        r=64, lora_alpha=128, lora_dropout=0.05, task_type='CAUSAL_LM',
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']))
    model.print_trainable_parameters()

def collate(batch):
    n = max(len(b['input_ids']) for b in batch)
    pad = tok.pad_token_id
    return {'input_ids': torch.tensor([b['input_ids'] + [pad] * (n - len(b['input_ids'])) for b in batch]),
            'labels': torch.tensor([b['labels'] + [-100] * (n - len(b['labels'])) for b in batch]),
            'attention_mask': torch.tensor([[1] * len(b['input_ids']) + [0] * (n - len(b['input_ids'])) for b in batch])}

args = TrainingArguments(
    output_dir=f'{WORK}/checkpoints-{NAME}', num_train_epochs=3, per_device_train_batch_size=8,
    gradient_accumulation_steps=2, learning_rate=1e-5 if FULL else 2e-4, lr_scheduler_type='cosine',
    warmup_ratio=0.03, weight_decay=0.0 if FULL else 0.01,
    logging_steps=20, eval_strategy='steps', eval_steps=200, save_steps=200, save_total_limit=2,
    bf16=bf16, fp16=not bf16, report_to='none', remove_unused_columns=False)
trainer = Trainer(model=model, args=args, train_dataset=train, eval_dataset=evals, data_collator=collate)
last = None
if os.path.isdir(args.output_dir) and any(d.startswith('checkpoint-') for d in os.listdir(args.output_dir)):
    last = True                                            # resume after a dropped session
trainer.train(resume_from_checkpoint=last)

In [ ]:
# Merge the adapter into the model (full fine-tuning has none) and save it.
merged = model if FULL else model.merge_and_unload()
merged.save_pretrained(f'{WORK}/merged-{NAME}', safe_serialization=True)
tok.save_pretrained(f'{WORK}/merged-{NAME}')
print('saved to', f'{WORK}/merged-{NAME}')

In [ ]:
# Try it on held-out messages: does it gather material, and does it stay on the words typed so far?
merged.eval()
for r in held_out[:4]:
    print('=' * 80)
    for prompt, gold in examples(r):
        ids = tok(prompt, return_tensors='pt', add_special_tokens=False).input_ids.to(merged.device)
        with torch.no_grad():
            out = merged.generate(ids, max_new_tokens=160, do_sample=False,
                                  eos_token_id=tok.convert_tokens_to_ids(END))
        print('>>', prompt[prompt.rindex('New words'):].strip())
        print(tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip() or '   (nothing)')

## Part 3: export to GGUF (restart the runtime first, then run the two setup cells)

In [ ]:
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip install -q gguf sentencepiece
name = STUDENT.split('/')[-1]
!python /content/llama.cpp/convert_hf_to_gguf.py {WORK}/merged-{name} --outtype q8_0 --outfile {WORK}/mafdet-gatherer-{name}-Q8_0.gguf
print(os.listdir(WORK))

Download `mafdet-gatherer-*-Q8_0.gguf` from the `mafdet-gatherer` folder of your Google Drive and put it in the
project folder. MAFDET will load it as the gatherer next to the large model.